# Assignment 06: Clinic Data Wrangling Pipeline (Midterm)

Follow `README.md`: it names every file to save in `output/`, with its first line and line count. Replace each **TODO**, then restart the kernel and **Run All** before you commit, so the saved outputs show your work.

In [ ]:
# Supplied setup: run this cell first and leave it unchanged.
from pathlib import Path
import hashlib
import json
import subprocess
import sys

import numpy as np
import pandas as pd

print("Python", sys.version.split()[0], "| NumPy", np.__version__, "| pandas", pd.__version__)


def find_assignment_dir(start):
    """The folder holding data_manifest.json: this one, a parent, or 06/mid-term below one."""
    for folder in (start, *start.parents):
        for candidate in (folder, folder / "06" / "mid-term"):
            if (candidate / "data_manifest.json").is_file():
                return candidate
    raise FileNotFoundError("Open this notebook from the assignment folder that holds data_manifest.json.")


BASE_DIR = find_assignment_dir(Path.cwd().resolve())
DATA_DIR = BASE_DIR / "data"
OUTPUT_DIR = BASE_DIR / "output"
OUTPUT_DIR.mkdir(exist_ok=True)
PROFILE_PATH = OUTPUT_DIR / "table_profile.csv"
AUDIT_PATH = OUTPUT_DIR / "issue_audit.csv"
PATIENTS_SUMMARY_PATH = OUTPUT_DIR / "patients_summary.csv"
GENDER_CATEGORIES_PATH = OUTPUT_DIR / "gender_categories.csv"
VISITS_SUMMARY_PATH = OUTPUT_DIR / "visits_summary.csv"
DECISION_LOG_PATH = OUTPUT_DIR / "decision_log.csv"
A1C_SUMMARY_PATH = OUTPUT_DIR / "a1c_summary.csv"
MERGE_AUDIT_PATH = OUTPUT_DIR / "merge_audit.csv"
SBP_WIDE_SAMPLE_PATH = OUTPUT_DIR / "sbp_wide_sample.csv"
ANALYSIS_SAMPLE_PATH = OUTPUT_DIR / "analysis_sample.csv"

# The patients whose rows the two extracts hold (README Tasks 4.3 and 5.2).
SAMPLE_IDS = ["P00001", "P00004", "P00088", "P00260", "P00446", "P00487", "P00631", "P00851",
              "P02354", "P03836", "P05763", "P07510", "P14287", "P16486", "P17390", "P30001"]

# The data is generated, not committed: generate_data.py writes the same tables on every computer.
manifest = json.loads((BASE_DIR / "data_manifest.json").read_text(encoding="utf-8"))
if not all((DATA_DIR / name).is_file() for name in manifest["files"]):
    print(subprocess.run([sys.executable, "generate_data.py"], cwd=BASE_DIR, check=True,
                         capture_output=True, text=True).stdout.strip())


def read_raw(path):
    """A Parquet file with the dtypes it stores; a CSV file with every column as text, blanks kept as written."""
    if path.suffix == ".parquet":
        return pd.read_parquet(path)
    return pd.read_csv(path, dtype="string", keep_default_na=False)


raw = {Path(name).stem: read_raw(DATA_DIR / name) for name in manifest["files"]}
# The checksum covers each table's contents, written out as CSV text.
for name, expected in manifest["files"].items():
    text = raw[Path(name).stem].to_csv(index=False, lineterminator="\n")
    assert hashlib.sha256(text.encode("utf-8")).hexdigest() == expected["sha256"], (
        f"data/{name} differs from the generated file; delete the data folder and run this cell again.")
print(f"Verified {len(manifest['files'])} data files")
raw_snapshot = {name: table.copy(deep=True) for name, table in raw.items()}
list(raw)

Python 3.13.15 | NumPy 2.3.3 | pandas 3.0.5
Verified 5 data files


['patients', 'sites', 'visits_north', 'visits_south', 'a1c_wide']

## Cumulative midterm checkpoint (human review)

lecture one: I used wc -l in 1.1 to count the lines in the file and saved the results to output/file_inventory.txt.

lecture two: I used summarize_table() in 1.2 to count the rows, columns, and duplicate rows in each dataset.

lecture three: I used NumPy after 4.3 to calculate the average systolic blood pressure at baseline (143.01 mmHg) and at week12 (137.15 mmHg) for 27,149 patients. I used axis=0 to calculate the average of each column.

lecture four: I used pandas DataFrames to see the datasets using .head(), .shape, and .dtypes before cleaning it.

lecture five: I used copy(deep=True) in 3.1 to make sure the raw patient data doesn't change while it was being cleaned, and I used validation checks in 5.1 to make sure the cleaned data followed the rules.

lecture six: I used pd.concat() in 2.2 to combine the North and South visit files into one table.

## Task 1: Inventory the files

### 1.1 Count lines from the terminal

Make `output/file_inventory.txt` in the terminal (README Task 1.1), then paste the command you ran into the `%%bash` cell below.

In [49]:
%%bash
wc -l data/*.csv > output/file_inventory.txt

### 1.2 Profile each table with a function

In [50]:
def summarize_table(name, table):

    num_duplicates = table.duplicated().sum()

    if num_duplicates > 0:
        duplicate_count = int(num_duplicates)
    else:
        duplicate_count = 0

    return {
        "table": name,
        "rows": len(table),
        "columns": len(table.columns),
        "exact_duplicate_rows": duplicate_count
    }

results = []

for name in manifest["files"]:
    results.append(summarize_table(name, raw[Path(name).stem]))

table_profile = pd.DataFrame(results)

table_profile.to_csv(
    OUTPUT_DIR / "table_profile.csv",
    index=False
)

display(table_profile)





,table,rows,columns,exact_duplicate_rows
0,patients.parquet,30045,4,45
1,sites.csv,5,3,0
2,visits_north.csv,26297,6,35
3,visits_south.csv,31074,6,35
4,a1c_wide.parquet,29424,3,0


## Task 2: Contract, stack, and audit

### 2.1 Write the data contract (human review)

grains and keys: Patients: Each row is one registered patient. The unique key is patient_id. Sites: Each row is one current clinic site. The unique key is site_code. Visits North: each row is one visit for a patient. The composite key is patient_id and visit. Visits South: each row is one visit for a patient. The composite key is patient_id and visit. A1c Wide: Each row is one patient's baseline and week12 A1c results. The unique key is patient_id

The patients table contains rows that have the same info. The sites table contains current and old records with the same site_code. The visit files contain duplicate rows and repeated patient-visit pairs becuase of rechecks. To fix this, duplicates are removed and kept the later recheck for each patient and visit combination.

Cardinality: Visits to Patients: Many-to-one because one patient can have multiple visits, but each visit belongs to one patient. Patients to Sites: Many-to-one because multiple patients can have to the same clinic site. Visits to Long-form A1c: One-to-one because each patient and visit combination can have only one A1c result.



### 2.2 Stack the two visit files

In [51]:
north = raw["visits_north"].copy(deep=True)
south = raw["visits_south"].copy(deep=True)

south = south.rename(columns={"wt_kg": "weight_kg"})

north["source_file"] = "visits_north.csv"
south["source_file"] = "visits_south.csv"

visits_raw = pd.concat([north, south], ignore_index=True)

display(visits_raw.head())
print(visits_raw.shape)

,patient_id,visit,visit_date,visit_time,sbp_mmhg,weight_kg,source_file
0,P00002,baseline,2026-02-20,09:50,132,85.2,visits_north.csv
1,P00002,week12,2026-05-08,15:20,129,82.0,visits_north.csv
2,P00003,baseline,2026-01-16,11:45,148,67.5,visits_north.csv
3,P00003,wk12,2026-04-07,10:00,137,64.0,visits_north.csv
4,P00004,baseline,2026-02-25,10:55,145,106.2,visits_north.csv


(57371, 7)


### 2.3 Audit the raw tables

In [52]:


patients = raw["patients"]
sites = raw["sites"]
visits = visits_raw

gender = patients["gender"].str.strip()

gender_not_code = (
    (gender != "") &
    (~patients["gender"].isin(["F", "M", "X"]))
).sum()

empty_gender = (gender == "").sum()

birth_year_9999 = (
    patients["birth_year"] == "9999"
).sum()

site_normalized = (
    patients["site_code"]
    .str.upper()
    .str.replace("-", "", regex=False)
    .str.replace(" ", "", regex=False)
)

site_needs_normalization = (
    patients["site_code"] != site_normalized
).sum()

repeated_sites = sites.duplicated(
    subset=["site_code"],
    keep=False
).sum()

invalid_visit_labels = (
    ~visits["visit"].isin(["baseline", "week12"])
).sum()

sbp = pd.to_numeric(
    visits["sbp_mmhg"],
    errors="coerce"
)

sbp_999 = (sbp == 999).sum()

sbp_outside_range = (
    (sbp != 999) &
    ((sbp < 70) | (sbp > 250))
).sum()

date_text = visits["visit_date"]

parsed_dates = pd.to_datetime(
    date_text,
    format="%Y-%m-%d",
    errors="coerce"
)

valid_dates = (
    parsed_dates.notna() &
    (parsed_dates.dt.strftime("%Y-%m-%d") == date_text)
)

invalid_dates = (~valid_dates).sum()

print(visits["visit"].value_counts())

normalized_visits = visits["visit"].str.strip().str.lower()
normalized_visits = normalized_visits.replace({
    "bl": "baseline",
    "wk12": "week12",
    "week 12": "week12"
})
normalized_visits = normalized_visits.where(
    normalized_visits.isin(["baseline", "week12"])
)

visit_pairs = pd.DataFrame({
    "patient_id": visits["patient_id"],
    "visit": normalized_visits
})

repeated_pairs = visit_pairs.duplicated(
    subset=["patient_id", "visit"],
    keep=False
).sum()

issue_audit = pd.DataFrame({
    "issue": [
        "patients: gender values that are not a code",
        "patients: empty gender",
        "patients: birth_year sentinel 9999",
        "patients: site codes needing normalization",
        "sites: rows with a repeated site_code",
        "visits: visit labels that are not baseline or week12",
        "visits: sbp sentinel 999",
        "visits: sbp outside 70 through 250",
        "visits: date parse failures",
        "visits: rows in repeated patient_id and visit pairs"
    ],
    "count": [
        gender_not_code,
        empty_gender,
        birth_year_9999,
        site_needs_normalization,
        repeated_sites,
        invalid_visit_labels,
        sbp_999,
        sbp_outside_range,
        invalid_dates,
        repeated_pairs
    ]
})

issue_audit["count"] = issue_audit["count"].astype(int)

issue_audit.to_csv(
    OUTPUT_DIR / "issue_audit.csv",
    index=False
)

display(issue_audit)



visit
baseline     28533
week12       26529
BASELINE       256
Baseline       242
bl             235
baseline       228
BL             220
wk12           204
 week12        198
Week 12        195
WEEK12         186
Wk12           177
week 12        168
Name: count, dtype: int64[pyarrow]


,issue,count
0,patients: gender values that are not a code,6968
1,patients: empty gender,486
2,patients: birth_year sentinel 9999,371
3,patients: site codes needing normalization,1811
4,sites: rows with a repeated site_code,4
5,visits: visit labels that are not baseline or ...,2309
6,visits: sbp sentinel 999,341
7,visits: sbp outside 70 through 250,116
8,visits: date parse failures,169
9,visits: rows in repeated patient_id and visit ...,380


## Task 3: Clean

### 3.1 Clean the patients

In [53]:


patients_clean = raw["patients"].copy(deep=True)

patients_clean = patients_clean.drop_duplicates()

print(patients_clean["gender"].value_counts(dropna=False))


patients_clean["gender"] = patients_clean["gender"].str.strip().str.lower()
patients_clean["gender"] = patients_clean["gender"].replace({
    "female": "f", "woman": "f",
    "male": "m", "man": "m",
    "nb": "x", "nonbinary": "x", "non-binary": "x"
})
patients_clean["gender"] = patients_clean["gender"].where(
    patients_clean["gender"].isin(["f", "m", "x"])
)
patients_clean["gender"] = patients_clean["gender"].str.upper().astype("category")

patients_clean["birth_year"] = (
    pd.to_numeric(patients_clean["birth_year"], errors="coerce")
    .replace(9999, pd.NA)
    .astype("Int64")
)

patients_clean["site_code"] = (
    patients_clean["site_code"]
    .str.upper()
    .str.replace("-", "", regex=False)
    .str.replace(" ", "", regex=False)
)

patients_clean = patients_clean.reset_index(drop=True)

patients_summary = pd.DataFrame({
    "metric": [
        "patients",
        "birth_year missing",
        "birth_year min",
        "birth_year max",
        "site_code values",
        "patients at N01",
        "patients at S02",
        "patients at E04"
    ],
    "value": [
        len(patients_clean),
        patients_clean["birth_year"].isna().sum(),
        patients_clean["birth_year"].min(),
        patients_clean["birth_year"].max(),
        patients_clean["site_code"].nunique(),
        (patients_clean["site_code"] == "N01").sum(),
        (patients_clean["site_code"] == "S02").sum(),
        (patients_clean["site_code"] == "E04").sum()
    ]
})

patients_summary.to_csv(
    OUTPUT_DIR / "patients_summary.csv",
    index=False
)

gender_counts = patients_clean["gender"].value_counts(dropna=False).reset_index()

gender_counts["code"] = gender_counts["gender"].cat.codes

gender_counts.to_csv(
    OUTPUT_DIR / "gender_categories.csv",
    index=False
)

display(patients_summary)
display(gender_counts)


gender
F             11230
M             10952
male           1038
female         1018
f               622
Male            621
Female          577
m               537
woman           399
                381
X               372
man             329
unknown         282
MALE            255
FEMALE          231
 M              223
 F              205
Woman           131
nonbinary       118
                104
NB              103
Unknown          96
non-binary       80
x                62
Non-Binary       34
Name: count, dtype: int64[pyarrow]


,metric,value
0,patients,30000
1,birth_year missing,371
2,birth_year min,1941
3,birth_year max,1996
4,site_code values,3
5,patients at N01,13739
6,patients at S02,15706
7,patients at E04,555


,gender,count,code
0,F,14413,0
1,M,13955,1
2,<NA>,863,-1
3,X,769,2


### 3.2 Clean the visits

In [54]:


visits_clean = visits_raw.copy(deep=True)

visits_clean = visits_clean.drop_duplicates()

print(visits_clean["visit"].value_counts())

visits_clean["visit"] = visits_clean["visit"].str.strip().str.lower()
visits_clean["visit"] = visits_clean["visit"].replace({
    "bl": "baseline",
    "wk12": "week12",
    "week 12": "week12"
})
visits_clean["visit"] = visits_clean["visit"].where(
    visits_clean["visit"].isin(["baseline", "week12"])
)

visits_clean["sbp_mmhg"] = pd.to_numeric(
    visits_clean["sbp_mmhg"], errors="coerce"
)
visits_clean["sbp_mmhg"] = visits_clean["sbp_mmhg"].where(
    visits_clean["sbp_mmhg"].between(70, 250)
).astype("Int64")

visits_clean["weight_kg"] = pd.to_numeric(
    visits_clean["weight_kg"],
    errors="coerce"
)

date_text = visits_clean["visit_date"]

parsed_dates = pd.to_datetime(
    date_text,
    format="%Y-%m-%d",
    errors="coerce"
)
valid_dates = parsed_dates.notna() & (
    parsed_dates.dt.strftime("%Y-%m-%d") == date_text
)
visits_clean["visit_date"] = parsed_dates.where(valid_dates)


visit
baseline     28496
week12       26498
BASELINE       255
Baseline       242
bl             235
baseline       228
BL             220
wk12           204
 week12        198
Week 12        194
WEEK12         186
Wk12           177
week 12        168
Name: count, dtype: int64[pyarrow]


In [55]:

rechecks = visits_clean[
    visits_clean.duplicated(
        subset=["patient_id", "visit"],
        keep=False
    )
]

display(rechecks)

visits_clean = (
    visits_clean
    .sort_values("visit_time")
    .drop_duplicates(
        subset=["patient_id", "visit"],
        keep="last"
    )
)


,patient_id,visit,visit_date,visit_time,sbp_mmhg,weight_kg,source_file
4,P00004,baseline,2026-02-25,10:55,145,106.2,visits_north.csv
5,P00004,baseline,2026-02-25,11:15,142,106.2,visits_north.csv
84,P00088,baseline,2026-03-10,09:30,153,68.2,visits_north.csv
85,P00088,baseline,2026-03-10,09:50,152,68.2,visits_north.csv
1208,P01341,week12,2026-05-26,15:10,121,72.3,visits_north.csv
...,...,...,...,...,...,...,...
55282,P28051,week12,2026-04-16,09:35,126,92.7,visits_south.csv
55472,P28232,baseline,2026-01-12,10:00,153,61.0,visits_south.csv
55473,P28232,baseline,2026-01-12,10:10,159,61.0,visits_south.csv
56248,P28953,baseline,2026-01-21,10:45,151,74.8,visits_south.csv


In [56]:

visits_clean = (
    visits_clean
    .sort_values(["patient_id", "visit"])
    .reset_index(drop=True)
)

visits_clean = visits_clean[
    ["patient_id", "visit", "visit_date",
     "sbp_mmhg", "weight_kg", "source_file"]
]

visits_summary = pd.DataFrame({
    "metric": [
        "visits",
        "baseline visits",
        "week12 visits",
        "visit_date missing",
        "visit_date min",
        "visit_date max",
        "sbp_mmhg missing",
        "sbp_mmhg min",
        "sbp_mmhg max",
        "weight_kg missing",
        "weight_kg mean",
        "visits from visits_north.csv",
        "visits from visits_south.csv"
    ],
    "value": [
        len(visits_clean),
        (visits_clean["visit"] == "baseline").sum(),
        (visits_clean["visit"] == "week12").sum(),
        visits_clean["visit_date"].isna().sum(),
        visits_clean["visit_date"].min(),
        visits_clean["visit_date"].max(),
        visits_clean["sbp_mmhg"].isna().sum(),
        visits_clean["sbp_mmhg"].min(),
        visits_clean["sbp_mmhg"].max(),
        visits_clean["weight_kg"].isna().sum(),
        round(visits_clean["weight_kg"].mean(), 2),
        (visits_clean["source_file"] == "visits_north.csv").sum(),
        (visits_clean["source_file"] == "visits_south.csv").sum()
    ]
})

visits_summary.to_csv(
    VISITS_SUMMARY_PATH,
    index=False
)

display(visits_summary)
print(visits_clean["visit"].value_counts(dropna=False))


,metric,value
0,visits,57181
1,baseline visits,29599
2,week12 visits,27582
3,visit_date missing,169
4,visit_date min,2026-01-05 00:00:00
5,visit_date max,2026-07-06 00:00:00
6,sbp_mmhg missing,456
7,sbp_mmhg min,70
8,sbp_mmhg max,250
9,weight_kg missing,828


visit
baseline    29599
week12      27582
Name: count, dtype: int64[pyarrow]


### 3.3 Record the decisions

In [68]:

decision_table = pd.DataFrame({
    "table": [
        "patients",
        "patients",
        "patients",
        "patients",
        "visits",
        "visits",
        "visits",
        "visits",
        "sites"
    ],
    "issue": [
        "exact duplicate rows",
        "gender spellings that are not a code",
        "site codes needing normalization",
        "birth_year sentinel 9999",
        "weight column named wt_kg in visits_south.csv",
        "sbp sentinel 999 and readings outside 70 through 250",
        "impossible and loosely written visit dates",
        "repeated patient_id and visit pairs",
        "repeated site_code rows"
    ],
    "decision": [
        "I removed duplicate rows and kept the first one.",
        "I converted gender values to F, M, or X and made the unknown values missing.",
        "I changed site codes to uppercase and removed spaces and hyphens.",
        "I replaced 9999 with missing values.",
        "I renamed wt_kg to weight_kg.",
        "I replaced invalid blood pressure readings with missing values.",
        "I converted valid dates and made invalid dates missing.",
        "I kept the later visit reading for repeated pairs.",
        "I kept only current records."
    ],
    "reason": [
        "Duplicate rows repeat the same patient information and alter the data and make it inaccurate.",
        "having all the same gender codes makes the data easier to analyze.",
        "having a standardized site codes allow the tables to match properly.",
        "9999 represents is a missing birth year and not a real year.",
        "Both visit files need the same column names before combining.",
        "Invalid blood pressure values should not be used in the analysis.",
        "I did this so everything would be consistent.",
        "The later reading is the recheck for that visit.",
        "This is so there weren't mutiple rows with the same info."
    ]
})

decision_table.to_csv(
    DECISION_LOG_PATH,
    index=False
)

display(decision_table)


,table,issue,decision,reason
0,patients,exact duplicate rows,I removed duplicate rows and kept the first one.,Duplicate rows repeat the same patient informa...
1,patients,gender spellings that are not a code,"I converted gender values to F, M, or X and ma...",having all the same gender codes makes the dat...
2,patients,site codes needing normalization,I changed site codes to uppercase and removed ...,having a standardized site codes allow the tab...
3,patients,birth_year sentinel 9999,I replaced 9999 with missing values.,9999 represents is a missing birth year and no...
4,visits,weight column named wt_kg in visits_south.csv,I renamed wt_kg to weight_kg.,Both visit files need the same column names be...
5,visits,sbp sentinel 999 and readings outside 70 throu...,I replaced invalid blood pressure readings wit...,Invalid blood pressure values should not be us...
6,visits,impossible and loosely written visit dates,I converted valid dates and made invalid dates...,I did this so everything would be consistent.
7,visits,repeated patient_id and visit pairs,I kept the later visit reading for repeated pa...,The later reading is the recheck for that visit.
8,sites,repeated site_code rows,I kept only current records.,This is so there weren't mutiple rows with the...


## Task 4: Reshape and join

### 4.1 Melt the A1c results to long form

In [58]:

a1c_long = raw["a1c_wide"].copy(deep=True)

a1c_long = a1c_long.melt(
    id_vars=["patient_id"],
    value_vars=["a1c_baseline", "a1c_week12"],
    var_name="visit",
    value_name="a1c_pct"
)

visit_map_a1c = {
    "a1c_baseline": "baseline",
    "a1c_week12": "week12"
}

a1c_long["visit"] = a1c_long["visit"].map(visit_map_a1c)

print(a1c_long["a1c_pct"].dtype)

a1c_long = (
    a1c_long
    .dropna(subset=["a1c_pct"])
    .reset_index(drop=True)
)

baseline = a1c_long[a1c_long["visit"] == "baseline"]
week12 = a1c_long[a1c_long["visit"] == "week12"]

a1c_summary = pd.DataFrame({
    "visit": ["baseline", "week12"],
    "results": [
        len(baseline),
        len(week12)
    ],
    "a1c_pct_mean": [
        round(baseline["a1c_pct"].mean(), 2),
        round(week12["a1c_pct"].mean(), 2)
    ]
})

a1c_summary.to_csv(
    A1C_SUMMARY_PATH,
    index=False
)

display(a1c_summary)


float64


,visit,results,a1c_pct_mean
0,baseline,29107,8.15
1,week12,27653,7.63


### 4.2 Check the keys before joining

In [59]:

sites = raw["sites"]

try:
    patients_clean.merge(
        sites,
        on="site_code",
        how="left",
        validate="many_to_one"
    )
except pd.errors.MergeError as e:
    print(e)

repeated_sites = sites[
    sites.duplicated(
        subset=["site_code"],
        keep=False
    )
]

display(repeated_sites)

sites_current = sites[
    sites["record_status"] == "current"
][["site_code", "site_name"]].copy()

display(sites_current)


Merge keys are not unique in right dataset; not a many-to-one merge

Duplicates in right:
 site_code
      N01
      S02 ...


,site_code,site_name,record_status
0,N01,North Clinic,current
1,N01,North Street Clinic,retired
2,S02,Southgate Clinic,retired
3,S02,South Valley Clinic,current


,site_code,site_name
0,N01,North Clinic
3,S02,South Valley Clinic
4,W03,West Clinic


In [60]:

visit_a1c_merge = visits_clean.merge(
    a1c_long,
    on=["patient_id", "visit"],
    how="outer",
    indicator=True
)

print(visit_a1c_merge["_merge"].value_counts())

merge_audit = pd.DataFrame({
    "check": [
        "visits with an A1c result",
        "visits without an A1c result",
        "A1c results without a visit",
        "visits whose patient is not in the registry",
        "registry patients with no visit",
        "registry patients whose site is not in the site list",
        "site list rows removed by the current-record rule"
    ],
    "count": [
        (visit_a1c_merge["_merge"] == "both").sum(),
        (visit_a1c_merge["_merge"] == "left_only").sum(),
        (visit_a1c_merge["_merge"] == "right_only").sum(),
        (~visits_clean["patient_id"].isin(
            patients_clean["patient_id"]
        )).sum(),
        (~patients_clean["patient_id"].isin(
            visits_clean["patient_id"]
        )).sum(),
        (~patients_clean["site_code"].isin(
            sites["site_code"]
        )).sum(),
        len(sites) - len(sites_current)
    ]
})

merge_audit["count"] = merge_audit["count"].astype(int)

merge_audit.to_csv(
    MERGE_AUDIT_PATH,
    index=False
)

display(merge_audit)


_merge
both          54006
left_only      3175
right_only     2754
Name: count, dtype: int64


,check,count
0,visits with an A1c result,54006
1,visits without an A1c result,3175
2,A1c results without a visit,2754
3,visits whose patient is not in the registry,79
4,registry patients with no visit,441
5,registry patients whose site is not in the sit...,555
6,site list rows removed by the current-record rule,2


### 4.3 Pivot blood pressure to one row per patient

In [61]:

sbp_wide = visits_clean.pivot(
    index="patient_id",
    columns="visit",
    values="sbp_mmhg"
)

sbp_wide = sbp_wide.reset_index()

sbp_wide.columns.name = None

sbp_wide["sbp_change"] = (
    sbp_wide["week12"] - sbp_wide["baseline"]
)

display(sbp_wide.head())

print(sbp_wide.columns)

sbp_wide[
    sbp_wide["patient_id"].isin(SAMPLE_IDS)
].to_csv(
    SBP_WIDE_SAMPLE_PATH,
    index=False
)


,patient_id,baseline,week12,sbp_change
0,P00001,146,129,-17
1,P00002,132,129,-3
2,P00003,148,137,-11
3,P00004,142,<NA>,<NA>
4,P00005,139,127,-12


Index(['patient_id', 'baseline', 'week12', 'sbp_change'], dtype='string')


#### NumPy checkpoint (Lecture 03, human review)

From the full `sbp_wide`, select `baseline` and `week12` in that order, keeping only patients with both readings. Name this two-column table `complete_pairs`; `np.array(complete_pairs, dtype=float)` converts it to a NumPy array. Display its shape and calculate one NumPy mean per visit across patients. Save the outputs and explain your axis choice in the cumulative checkpoint's Lecture 03 bullet. No extra output file.

In [62]:

complete_pairs = sbp_wide[
    ["baseline", "week12"]
].dropna()

sbp_array = np.array(
    complete_pairs,
    dtype=float
)

print(sbp_array.shape)

visit_means = np.mean(
    sbp_array,
    axis=0
)

print(visit_means)


(27149, 2)
[143.01337066 137.15061328]


### 4.4 Join the analysis table

In [63]:


analysis = visits_clean.copy(deep=True)

analysis = analysis.merge(
    patients_clean,
    on="patient_id",
    how="left",
    validate="many_to_one"
)

analysis = analysis.merge(
    sites_current,
    on="site_code",
    how="left",
    validate="many_to_one"
)

analysis = analysis.merge(
    a1c_long,
    on=["patient_id", "visit"],
    how="left",
    validate="one_to_one"
)

analysis = analysis[
    [
        "patient_id",
        "visit",
        "visit_date",
        "sbp_mmhg",
        "weight_kg",
        "a1c_pct",
        "gender",
        "birth_year",
        "site_code",
        "site_name",
        "source_file"
    ]
]

analysis = analysis.astype({
    "patient_id": "string",
    "visit": "string",
    "sbp_mmhg": "Int64",
    "weight_kg": "float64",
    "a1c_pct": "float64",
    "gender": "string",
    "birth_year": "Int64",
    "site_code": "string",
    "site_name": "string",
    "source_file": "string"
})

display(analysis.head())
print(analysis.columns.tolist())
print(analysis.dtypes)
print(analysis.shape)


,patient_id,visit,visit_date,sbp_mmhg,weight_kg,a1c_pct,gender,birth_year,site_code,site_name,source_file
0,P00001,baseline,2026-03-06,146,52.4,7.3,F,1966,S02,South Valley Clinic,visits_south.csv
1,P00001,week12,2026-05-27,129,50.2,NaN,F,1966,S02,South Valley Clinic,visits_south.csv
2,P00002,baseline,2026-02-20,132,85.2,8.4,F,<NA>,N01,North Clinic,visits_north.csv
3,P00002,week12,2026-05-08,129,82.0,7.4,F,<NA>,N01,North Clinic,visits_north.csv
4,P00003,baseline,2026-01-16,148,67.5,5.9,F,1970,N01,North Clinic,visits_north.csv


['patient_id', 'visit', 'visit_date', 'sbp_mmhg', 'weight_kg', 'a1c_pct', 'gender', 'birth_year', 'site_code', 'site_name', 'source_file']
patient_id             string
visit                  string
visit_date     datetime64[us]
sbp_mmhg                Int64
weight_kg             float64
a1c_pct               float64
gender                 string
birth_year              Int64
site_code              string
site_name              string
source_file            string
dtype: object
(57181, 11)


## Task 5: Validate, save, and read back

### 5.1 Validate before saving

In [64]:


checks = pd.Series({
    "analysis row count": len(analysis) == len(visits_clean),
    "unique patient visit pairs": not analysis.duplicated(
        subset=["patient_id", "visit"]
    ).any(),
    "valid visit labels": analysis["visit"].isin(
        ["baseline", "week12"]
    ).all(),
    "valid sbp values": analysis["sbp_mmhg"].dropna().between(
        70, 250
    ).all(),
    "unique patient keys": patients_clean["patient_id"].is_unique,
    "unique site keys": sites_current["site_code"].is_unique,
    "unique a1c keys": not a1c_long.duplicated(
        subset=["patient_id", "visit"]
    ).any(),
    "current site names": analysis["site_name"].dropna().isin(
        sites_current["site_name"]
    ).all(),
    "unique sbp patient IDs": sbp_wide["patient_id"].is_unique,
    "raw tables unchanged": all(
        raw[name].equals(raw_snapshot[name])
        for name in raw
    )
})

display(checks)
print(checks[~checks])
assert checks.all(), checks[~checks]


analysis row count            True
unique patient visit pairs    True
valid visit labels            True
valid sbp values              True
unique patient keys           True
unique site keys              True
unique a1c keys               True
current site names            True
unique sbp patient IDs        True
raw tables unchanged          True
dtype: bool

Series([], dtype: bool)


### 5.2 Save the analysis extract

In [65]:

analysis_sample = analysis[
    analysis["patient_id"].isin(SAMPLE_IDS)
].copy()

analysis_sample = analysis_sample.reset_index(drop=True)

analysis_sample.to_csv(
    ANALYSIS_SAMPLE_PATH,
    index=False
)

display(analysis_sample)


,patient_id,visit,visit_date,sbp_mmhg,weight_kg,a1c_pct,gender,birth_year,site_code,site_name,source_file
0,P00001,baseline,2026-03-06,146,52.4,7.3,F,1966,S02,South Valley Clinic,visits_south.csv
1,P00001,week12,2026-05-27,129,50.2,NaN,F,1966,S02,South Valley Clinic,visits_south.csv
2,P00004,baseline,2026-02-25,142,106.2,7.3,X,<NA>,N01,North Clinic,visits_north.csv
3,P00088,baseline,2026-03-10,152,68.2,8.4,F,1980,N01,North Clinic,visits_north.csv
4,P00088,week12,2026-06-03,147,66.3,7.6,F,1980,N01,North Clinic,visits_north.csv
5,P00260,baseline,NaT,70,80.4,9.5,F,1979,E04,<NA>,visits_south.csv
6,P00260,week12,2026-06-10,138,78.6,8.0,F,1979,E04,<NA>,visits_south.csv
7,P00446,baseline,NaT,250,NaN,6.5,F,1946,S02,South Valley Clinic,visits_south.csv
8,P00446,week12,2026-04-02,144,94.3,6.8,F,1946,S02,South Valley Clinic,visits_south.csv
9,P00487,baseline,2026-03-02,<NA>,103.0,7.9,M,1959,S02,South Valley Clinic,visits_south.csv


### 5.3 Read the file back

In [66]:

analysis_readback = pd.read_csv(
    ANALYSIS_SAMPLE_PATH,
    dtype={
        "patient_id": "string",
        "visit": "string",
        "sbp_mmhg": "Int64",
        "weight_kg": "float64",
        "a1c_pct": "float64",
        "gender": "string",
        "birth_year": "Int64",
        "site_code": "string",
        "site_name": "string",
        "source_file": "string"
    },
    parse_dates=["visit_date"]
)

analysis_readback["visit_date"] = analysis_readback["visit_date"].astype(
    analysis_sample["visit_date"].dtype
)

print(analysis_readback.dtypes)
print(analysis_readback.equals(analysis_sample))

patient_id             string
visit                  string
visit_date     datetime64[us]
sbp_mmhg                Int64
weight_kg             float64
a1c_pct               float64
gender                 string
birth_year              Int64
site_code              string
site_name              string
source_file            string
dtype: object
True
